# Multiple-Choice QA Experiments

This notebook contains the code used for:

- RACE-M
- ARC-Challenge
- OpenBookQA

## How to use the notebook

1. Run the shared import, API, model, and answer-mapping cells.
2. Run the **loading and label-extraction cell for only one dataset**.
3. Run the **prompt cell for that same dataset**.
4. Run the shared DeepInfra helper.
5. Run the inference and evaluation cells for the selected dataset.

The variable names are intentionally kept the same across datasets:

- `example_ids`
- `correct_answers`
- `labels`
- `build_prompt1`
- `build_prompt2`

Therefore, running another dataset section overwrites these variables and prompt
functions. 

## 1. Shared imports

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

## 2. DeepInfra API configuration

In [ ]:
from google.colab import userdata
DEEPINFRA_TOKEN = userdata.get("deepinfra_apikey")

In [ ]:
from openai import OpenAI
client = OpenAI(
    api_key=DEEPINFRA_TOKEN,
    base_url="https://api.deepinfra.com/v1/openai",
)

## 3. Model and result configuration

In [ ]:

MODEL_NAME = "Qwen/Qwen3.5-27B"
SAFE_MODEL_NAME = MODEL_NAME.replace("/", "_")


## 5. RACE-M

In [ ]:
# This cell is only for loading RACE-M and extracting its labels.
# Run this cell before the RACE-M prompt cell.

from datasets import load_dataset
import numpy as np

race_m_test = load_dataset(
    "ehovy/race",
    "middle",
    split="test"
)

answer_to_idx = {"A": 0, "B": 1, "C": 2, "D": 3}
idx_to_answer = {0: "A", 1: "B", 2: "C", 3: "D"}

correct_answers = [ex["answer"] for ex in race_m_test]
labels = np.array([answer_to_idx[a] for a in correct_answers])

example_ids = list(range(len(race_m_test)))

# These variables are used by the shared inference and evaluation cells.
test_data = race_m_test
dataset_name = "RACE-M"


In [ ]:
# These prompts are for RACE-M because each question includes a passage.
# Run this cell only when evaluating RACE-M.

def build_prompt1(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]

    prompt = f"""

 Read the passage and answer the multiple-choice question correctly.

Passage:
{article}
Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt

def build_prompt2(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]

    prompt = f"""

 Read the passage and answer the multiple-choice question incorrectly.

Passage:
{article}
Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


def build_prompt1_v2(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]

    prompt = f"""

Read the passage, understand the context, and select the option that best answers the question.

Passage:
{article}
Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt

def build_prompt2_v2(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]

    prompt = f"""

Read the passage, understand the context, and intentionally select an incorrect option for the question.

Passage:
{article}
Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt

def build_prompt1_v3(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]

    prompt = f"""

Carefully read the passage and select the single option that correctly answers the question according to the passage.

Passage:
{article}
Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt

def build_prompt2_v3(example):
    article = example["article"]
    question = example["question"]
    options = example["options"]

    prompt = f"""

Carefully read the passage and determine which option correctly answers the question. Do not select that option. Instead, select an incorrect option.

Passage:
{article}
Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt

## Shared answer mappings for ARC-Challenge and OpenBookQA

In [ ]:
answer_to_idx = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "1": 0,
    "2": 1,
    "3": 2,
    "4": 3,
}

idx_to_answer = {
    0: "A",
    1: "B",
    2: "C",
    3: "D"
}

OPTIONS = ["A", "B", "C", "D"]

## 6. ARC-Challenge

In [ ]:
# This cell is used only for loading ARC-Challenge, keeping examples
# with exactly four choices, and extracting the labels.

arc_test = load_dataset(
    "allenai/ai2_arc",
    "ARC-Challenge",
    split="test",
)

# Remove examples that do not have exactly four answer choices.
arc_test_4choices = [
    example
    for example in arc_test
    if len(example["choices"]["text"]) == 4
]

print("Original ARC-Challenge size:", len(arc_test))
print("ARC-Challenge four-choice size:", len(arc_test_4choices))

example_ids = []
correct_answers = []
labels = []

for i, example in enumerate(arc_test_4choices):
    example_id = example.get("example_id", i)

    example_ids.append(example_id)
    correct_answers.append(example["answerKey"])
    labels.append(answer_to_idx[example["answerKey"]])

labels = np.array(labels)

# These variables are used by the shared inference and evaluation cells.
test_data = arc_test_4choices
dataset_name = "ARC-Challenge"


## 7. OpenBookQA

In [ ]:
# This cell is used only for loading OpenBookQA and extracting the labels.

openbook_test = load_dataset(
    "allenai/openbookqa",
    "main",
    split="test",
)

example_ids = []
correct_answers = []
labels = []

for i, example in enumerate(openbook_test):
    example_id = example.get("id", i)

    example_ids.append(example_id)
    correct_answers.append(example["answerKey"])
    labels.append(answer_to_idx[example["answerKey"]])

labels = np.array(labels)

# These variables are used by the shared inference and evaluation cells.
test_data = openbook_test
dataset_name = "OpenBookQA"


## 8. Prompts used for both ARC-Challenge and OpenBookQA

These prompt functions are used for both ARC-Challenge and OpenBookQA because
both datasets use a question with four choices and do not contain the passage
field used by RACE-M.

Run only one prompt-version cell before inference.

In [ ]:
# This cell is used for both ARC-Challenge and OpenBookQA.
# Prompt version 1.

def build_prompt1(example):
    question = example["question_stem"]
    options = example["choices"]["text"]

    prompt = f"""
Answer the multiple-choice question correctly.

Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


def build_prompt2(example):
    question = example["question_stem"]
    options = example["choices"]["text"]

    prompt = f"""
Answer the multiple-choice question incorrectly.

Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


In [ ]:
# This cell is used for both ARC-Challenge and OpenBookQA.
# Prompt version 2.

def build_prompt1(example):
    question = example["question_stem"]
    options = example["choices"]["text"]

    prompt = f"""
Select the option that best answers the question.

Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


def build_prompt2(example):
    question = example["question_stem"]
    options = example["choices"]["text"]

    prompt = f"""
Intentionally select an incorrect option for the question.

Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


In [ ]:
# This cell is used for both ARC-Challenge and OpenBookQA.
# Prompt version 3.

def build_prompt1(example):
    question = example["question_stem"]
    options = example["choices"]["text"]

    prompt = f"""
Select the single option that correctly answers the question.

Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


def build_prompt2(example):
    question = example["question_stem"]
    options = example["choices"]["text"]

    prompt = f"""
Carefully determine which option correctly answers the question.
Do not select that option. Instead, select an incorrect option.

Question:
{question}

Options:
A. {options[0]}
B. {options[1]}
C. {options[2]}
D. {options[3]}

Return only one letter: A, B, C, or D.

Answer:
"""
    return prompt


## 9. Shared DeepInfra inference helper

In [ ]:
import time
import numpy as np
from tqdm import tqdm
from openai import OpenAI

valid_choices = ["A", "B", "C", "D"]


def normalize_token_text(token):
    if token is None:
        return ""

    token = token.strip()
    token = token.replace("Ġ", "")
    token = token.replace("▁", "")
    token = token.strip()
    return token.upper()


def call_deepinfra_one_token(prompt, max_retries=5, sleep_time=3):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=messages,
                temperature=0,
                max_tokens=10,
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                }
            )

            raw_output = response.choices[0].message.content
            generated_token = normalize_token_text(raw_output)
            if generated_token in valid_choices:
                prediction_idx = answer_to_idx[generated_token]
            else:
                prediction_idx = -1

            return {
                "prediction_idx": prediction_idx,
                "generated_token": generated_token,
                "raw_output": raw_output,
                "error": None
            }

        except Exception as e:
            print(f"[Retry {attempt + 1}/{max_retries}] Error: {e}")
            time.sleep(sleep_time)

    return {
        "prediction_idx": -1,
        "generated_token": "",
        "raw_output": None,
        "error": "Max retries failed"
    }

## 10. Shared inference

Run the loading cell and prompt cell for the dataset you want to test. To test
another model, change `MODEL_NAME` in the model-configuration cell. The same
loop is used for all datasets.

In [ ]:
# Change the dataset by running its loading/label cell and prompt cell.
# Change MODEL_NAME whenever you want to evaluate another model.

standard_predictions = []
non_standard_predictions = []

standard_results = []
non_standard_results = []

for example in tqdm(
    test_data,
    desc=f"Running {MODEL_NAME} on {dataset_name}",
):
    prompt_standard = build_prompt1(example)
    prompt_non_standard = build_prompt2(example)

    standard_result = call_deepinfra_one_token(prompt_standard)
    non_standard_result = call_deepinfra_one_token(prompt_non_standard)

    standard_results.append(standard_result)
    non_standard_results.append(non_standard_result)

    standard_predictions.append(standard_result["prediction_idx"])
    non_standard_predictions.append(non_standard_result["prediction_idx"])

standard_predictions = np.array(standard_predictions)
non_standard_predictions = np.array(non_standard_predictions)


## 11. Count invalid outputs

In [ ]:
standard_not_abcd = standard_predictions == -1
non_standard_not_abcd = non_standard_predictions == -1

num_standard_not_abcd = standard_not_abcd.sum()
num_non_standard_not_abcd = non_standard_not_abcd.sum()

print("Standard generated token not in A/B/C/D:", num_standard_not_abcd)
print("Non-standard generated token not in A/B/C/D:", num_non_standard_not_abcd)


## 12. Shared evaluation

The same evaluation code is used for all datasets. To evaluate another dataset
or model, run the corresponding dataset and prompt cells, change `MODEL_NAME`
if needed, and rerun inference and evaluation.

In [ ]:
# This evaluation cell is shared by RACE-M, ARC-Challenge, and OpenBookQA.

standard_correct = standard_predictions == labels
non_standard_correct = non_standard_predictions == labels

correct_in_standard = standard_correct.sum()
correct_in_non_standard = non_standard_correct.sum()
correct_in_both = (standard_correct & non_standard_correct).sum()

standard_accuracy = (correct_in_standard / len(test_data)) * 100
non_standard_accuracy = (correct_in_non_standard / len(test_data)) * 100

if correct_in_standard > 0:
    iffr = (correct_in_both / correct_in_standard) * 100
else:
    iffr = 0.0

print("Dataset:", dataset_name)
print("Model:", MODEL_NAME)
print("Standard invalid A/B/C/D count:", num_standard_not_abcd)
print("Non-standard invalid A/B/C/D count:", num_non_standard_not_abcd)

print("Correct in standard:", correct_in_standard)
print("Correct in non-standard:", correct_in_non_standard)
print("Correct in both:", correct_in_both)

print(f"Standard accuracy: {standard_accuracy:.2f}")
print(f"Non-standard accuracy: {non_standard_accuracy:.2f}")
print(f"IFFR: {iffr:.2f}")
